# mini GPT LLM 구현 — 실습 노트북

과제 문서 「과제 - GPT LLM 구현」의 Day 1~8 순서를 따릅니다.

1. 환경설정과 데이터 준비 셀을 실행합니다.
2. 현재 Day의 개념과 작은 입력을 먼저 확인합니다.
3. 해당 `src/` 파일의 TODO를 직접 구현합니다.
4. 현재 Day의 테스트와 `selfcheck`를 실행합니다.
5. 구현한 계산의 원리, 결과, 핵심 불변조건을 설명할 수 있으면 다음 Day로 이동합니다.

`NotImplementedError`가 나오면 현재 Day의 TODO가 남아 있는지 확인하세요. docstring과 테스트 메시지로 막히면 `HINTS.md`를 1단계부터 하나씩 여세요.

## 0. 환경설정

In [ ]:
# Colab에서는 이 셀을 가장 먼저 실행하세요. 로컬에서는 현재 폴더(프로젝트 루트)를 사용합니다.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not Path('src').exists():
    url = input('저장소 URL (예: https://github.com/USERNAME/gpt-lab-v2): ').strip()
    tok = input('GitHub Personal Access Token (공개 저장소면 Enter): ').strip()
    url = url.replace('https://', '').replace('http://', '').rstrip('/')
    repo_dir = Path(url.split('/')[-1].removesuffix('.git'))
    if not repo_dir.exists():
        remote = f'https://{tok}@{url}' if tok else f'https://{url}'
        subprocess.run(['git', 'clone', remote, str(repo_dir)], check=True)
    os.chdir(repo_dir)

ROOT = Path.cwd().resolve()
assert (ROOT / 'src').exists(), f'{ROOT}에 src/가 없습니다. 프로젝트 루트에서 노트북을 실행하세요.'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('ROOT =', ROOT)

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

In [ ]:
# src/ 파일을 고치면 커널을 재시작하지 않아도 다시 불러옵니다.
%load_ext autoreload
%autoreload 2

import math, time
import torch
from utils import set_seed, get_device, count_parameters, model_summary, shape_trace
from configs import (SMOKE_CONFIG, LIGHT_CONFIG, BASIC_CONFIG,
                     SMOKE_TRAIN, LIGHT_TRAIN, BASIC_TRAIN,
                     SMOKE_DATA, LIGHT_DATA, BASIC_DATA, assert_vocab_match)

set_seed(42)
DEVICE = get_device()
print('python:', sys.version.split()[0], '| torch:', torch.__version__, '| device:', DEVICE)

In [ ]:
# 데이터 준비 (처음 한 번만 다운로드합니다)
if not Path('data/nsmc_lm_train.txt').exists():
    import download_data
    download_data.main()

corpus = Path('data/nsmc_lm_train.txt').read_text(encoding='utf-8')
val_corpus = Path('data/nsmc_lm_val.txt').read_text(encoding='utf-8')
print(f'사전 학습 train {len(corpus):,}자 / val {len(val_corpus):,}자')

In [ ]:
# 단계별 테스트와 selfcheck 실행 helper
def run_pytest(*targets, k=None):
    cmd = [sys.executable, '-m', 'pytest', *(targets or ['tests/']), '-q', '--no-header',
           '-p', 'no:cacheprovider']
    if k:
        cmd += ['-k', k]
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(r.stdout[-3000:] or r.stderr[-3000:])

def selfcheck(day=None, trace=False):
    cmd = [sys.executable, 'tools/selfcheck.py'] + (['--day', str(day)] if day else ['--all'])
    if trace:
        cmd.append('--trace')
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(r.stdout or r.stderr)

---
## Day 1 — LLM과 GPT 기본 개념

**먼저 읽을 교재**: 1장 전체(1.1~1.8)

핵심 질문: LLM은 무엇을 어떻게 학습하며, GPT는 어떤 구조로 한 token씩 문장을 만드는가?

이 Day에는 TODO와 테스트가 없습니다. 위의 환경설정과 데이터 준비 셀을 실행한 뒤 다음을 진행합니다.

1. 교재 1장을 과제 문서 7.1.1절의 확인할 것을 짚으며 읽습니다.
2. 과제 문서 7.1.2절의 참고 자료로 개념을 보충합니다. 먼저 볼 자료:
   - [3Blue1Brown, Large Language Models explained briefly](https://www.youtube.com/watch?v=LPZh9BOjkQs) (약 8분)
   - [3Blue1Brown, Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M) (약 27분)
   - [Andrej Karpathy, Intro to Large Language Models](https://www.youtube.com/watch?v=zjkBMFhNj_g) (약 1시간)
3. [Transformer Explainer](https://poloclub.github.io/transformer-explainer/)에 짧은 문장을 넣고 다음 token 후보와 확률, temperature를 바꿨을 때의 변화를 관찰해 `REPORT.md` 1장에 메모합니다.
4. 아래 셀로 개념 질문을 확인하고 팀원끼리 코드 없이 답해 봅니다.

교재 1.7의 3단계는 이 노트북에서 다음처럼 이어집니다: 1단계 LLM 구현(Day 2~7), 2단계 사전 학습(Day 7~8), 3단계 분류 미세 튜닝(Day 8).

In [ ]:
selfcheck(1)

---
## Day 2 — byte-level BPE

**먼저 읽을 교재**: 2.1~2.5 (2.5의 `tiktoken` 대신 byte-level BPE를 직접 구현)

핵심 질문: 문자열을 왜 그대로 모델에 넣을 수 없으며, BPE는 무엇을 학습하는가?

**구현**: `src/bpe.py`의 `get_pair_counts`, `merge_pair`, `BPETokenizer.train`의 병합 루프

먼저 아래 결과를 손으로 예상한 뒤 코드와 비교하세요.

In [ ]:
from bpe import get_pair_counts, merge_pair
print(get_pair_counts([1, 2, 1, 2, 3]))          # {(1, 2): 2, (2, 1): 1, (2, 3): 1}
print(merge_pair([1, 2, 1, 2, 3], (1, 2), 99))   # [99, 99, 3]
print(merge_pair([5, 5, 5], (5, 5), 99))         # [99, 5]  겹쳐서 병합하지 않음

# train 병합 루프: 손으로 계산한 merge 순서와 비교 (a=101, b=102, c=103, d=104)
from bpe import BPETokenizer
toy = BPETokenizer(vocab_size=300).train('aaabdaaabac')
print(toy.merges)                       # [(101, 101), (101, 102), (260, 261)]
print([toy.id_to_token[i] for i in range(260, len(toy.id_to_token))])   # [b'aa', b'ab', b'aaab']
print(toy.encode('aaabdaaabac'))        # [262, 104, 262, 101, 103]

In [ ]:
run_pytest('tests/test_bpe.py')

In [ ]:
# Smoke vocabulary 학습과 round-trip 확인
from bpe import BPETokenizer

t0 = time.time()
tokenizer = BPETokenizer(vocab_size=SMOKE_DATA['vocab_size'])
tokenizer.train(corpus[:SMOKE_DATA['chars']])
# BPE 학습에서 막혔다면 위 두 줄 대신 제공된 Smoke vocabulary를 불러 Day 3로 진행하세요.
# tokenizer = BPETokenizer(vocab_size=300); tokenizer.load(SMOKE_DATA['vocab_path'])
print(f'{time.time()-t0:.2f}초, 어휘 {len(tokenizer.id_to_token)}개, merge {len(tokenizer.merges)}개')

samples = ['이 영화는 정말 좋았다!', '연기, 음악, 연출 모두 좋음 ㅋㅋ', '한글 English 123 !?']
for text in samples:
    assert tokenizer.decode(tokenizer.encode(text, add_bos_eos=True)) == text
print('round-trip OK')

In [ ]:
# 한글이 byte로 쪼개지고 merge token으로 합쳐지는 과정 관찰
s = '한글'
print('UTF-8 bytes   :', list(s.encode('utf-8')))
print('byte token IDs:', [b + 4 for b in s.encode('utf-8')], ' # byte 값 + 4')
print('BPE token IDs :', tokenizer.encode(s))
print('token bytes   :', [tokenizer.id_to_token[i] for i in tokenizer.encode(s)])

print('\nID 범위: 0~3 특수 token / 4~259 byte token / 260 이상 merge token')
for i in range(260, 275):
    b = tokenizer.id_to_token[i]
    print(i, b, repr(b.decode('utf-8', errors='replace')))

In [ ]:
selfcheck(2)

---
## Day 3 — Next-token Dataset과 Embedding

**먼저 읽을 교재**: 2.6~2.8

핵심 질문: input과 target은 왜 한 칸 차이며, 같은 token의 위치는 어떻게 표현하는가?

**구현**: `src/dataset.py`의 `GPTDatasetV1.__init__` 슬라이딩 윈도 루프

token·position embedding은 교재 2.7~2.8처럼 `nn.Embedding`으로 먼저 관찰하고, Day 7의 `GPTModel.forward`에서 구현합니다.

In [ ]:
run_pytest('tests/test_dataset.py')

In [ ]:
# 첫 batch를 decode해 input과 target의 한 칸 차이를 확인합니다
from dataset import GPTDatasetV1, create_dataloader_v1

loader = create_dataloader_v1(corpus[:2000], tokenizer, batch_size=2, max_length=16,
                              stride=16, shuffle=False)
xb, yb = next(iter(loader))
print('input  ids:', xb[0].tolist())
print('target ids:', yb[0].tolist())
print('input [1:] == target[:-1] ?', xb[0, 1:].tolist() == yb[0, :-1].tolist())
# window 경계에서 한글의 UTF-8 byte가 잘리면 decode 결과 양 끝에 '�'가 보일 수 있습니다.
print('input  text:', repr(tokenizer.decode(xb[0].tolist())))
print('target text:', repr(tokenizer.decode(yb[0].tolist())))

In [ ]:
# 교재 2.7~2.8: token embedding + position embedding
vocab_size, output_dim, context_length = SMOKE_CONFIG['vocab_size'], 8, 4
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

inputs = xb[:, :context_length]                                      # (B=2, T=4)
token_embeddings = token_embedding_layer(inputs)                     # (2, 4, 8)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))   # (4, 8)
input_embeddings = token_embeddings + pos_embeddings                 # broadcasting -> (2, 4, 8)
print('token', tuple(token_embeddings.shape), '| position', tuple(pos_embeddings.shape),
      '| input', tuple(input_embeddings.shape))

same = token_embedding_layer(torch.tensor([[7, 7, 7, 7]])) + pos_embeddings
print('위치가 다른 같은 token 7의 벡터가 같은가:', torch.allclose(same[0, 0], same[0, 1]), ' <- False여야 정상')

In [ ]:
selfcheck(3)

---
## Day 4 — Single-head scaled dot-product attention

**먼저 읽을 교재**: 3.1~3.4, 3.5.1

핵심 질문: 한 token이 다른 token을 얼마나 참고할지를 어떤 계산으로 정하는가?

**구현**: `src/attention_practice.py`의 `scaled_dot_product_attention`

```text
queries, keys, values  (B, T, D)
attn_scores           (B, T, T) = queries @ keys^T
attn_weights          (B, T, T) = softmax(mask(attn_scores) / sqrt(D))
context_vec           (B, T, D) = attn_weights @ values
```

`mask`는 True인 칸을 가리는 bool tensor입니다. 교재 3.5.1처럼 `torch.triu`로 직접 만들어 넘깁니다.

이 함수는 Day 5의 `MultiHeadAttention.forward`가 `(B, H, T, D)` 입력으로 그대로 호출합니다. 축은 `-1`, `-2`처럼 뒤에서부터 지정하세요.

Day 4에는 multi-head reshape를 넣지 않습니다. 아래 `T=3`, `D=2` 예제의 score와 softmax를 먼저 손으로 계산하세요.

In [ ]:
from attention_practice import scaled_dot_product_attention

x = torch.tensor([[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]])   # (B=1, T=3, D=2)
out, w = scaled_dot_product_attention(x, x, x)
print('score = q @ k^T / sqrt(2)')
print((x @ x.transpose(-2, -1) / math.sqrt(2))[0].round(decimals=3))
print('\nweight (각 행의 합이 1)')
print(w[0].round(decimals=3), '\n행 합:', w[0].sum(-1))
print('\nout')
print(out[0].round(decimals=3))

In [ ]:
# sqrt(D) scaling이 softmax 분포에 미치는 영향: D가 커지면 score 분산도 커집니다
torch.manual_seed(0)
D = 64
q, k = torch.randn(1, 5, D), torch.randn(1, 5, D)
raw = (q @ k.transpose(-2, -1))[0, 0]
print('scaled  :', torch.softmax(raw / math.sqrt(D), -1).round(decimals=3).tolist())
print('unscaled:', torch.softmax(raw, -1).round(decimals=3).tolist())

In [ ]:
# causal mask (교재 3.5.1): 대각선 위쪽(미래 위치)이 True인 bool 행렬을 만들어 넘깁니다
T = 5
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
print(mask.int())

x1 = torch.randn(1, T, 4)
x2 = x1.clone(); x2[0, 3:] = torch.randn(2, 4)
o1, w1 = scaled_dot_product_attention(x1, x1, x1, mask=mask)
o2, _ = scaled_dot_product_attention(x2, x2, x2, mask=mask)
print(w1[0].round(decimals=3))
print('과거(0~2) 출력이 동일한가:', torch.allclose(o1[0, :3], o2[0, :3], atol=1e-6))

In [ ]:
run_pytest('tests/test_attention_practice.py')
selfcheck(4)

---
## Day 5 — Causal Multi-Head Attention

**먼저 읽을 교재**: 3.5.2~3.5.3, 3.6(3.6.1~3.6.2)

핵심 질문: 여러 attention head가 서로 다른 관계를 학습하도록 계산을 어떻게 나누며, causal mask는 미래 정보 사용을 어떻게 막는가?

**구현**: `MultiHeadAttention.forward` — 생성자는 교재 3.6.2와 같고(`W_query`, `W_key`, `W_value`, `out_proj`, `mask` buffer), score·mask·softmax는 Day 4의 `scaled_dot_product_attention`을 호출합니다. 여기서는 head 분리·결합과 projection을 작성합니다.

```text
x                       (B, T, d_in)
queries, keys, values   (B, T, d_out)
head 분리                (B, H, T, D), d_out = H * D
Day 4 함수 호출          context_vec (B, H, T, D), attn_weights (B, H, T, T)
head 결합                (B, T, d_out)
```

In [ ]:
run_pytest('tests/test_attention.py')

In [ ]:
# mask 적용 전·후 attention matrix 비교
from attention import MultiHeadAttention

mha = MultiHeadAttention(d_in=8, d_out=8, context_length=16, dropout=0.0, num_heads=2)
mha.eval()
xx = torch.randn(1, 5, 8)
_, w_on = mha(xx, causal_mask=True, return_attention_weights=True)
_, w_off = mha(xx, causal_mask=False, return_attention_weights=True)
print('weight shape (B, H, T, T):', tuple(w_on.shape))
print('\ncausal_mask=True  (head 0) — 상삼각이 0이어야 합니다')
print(w_on[0, 0].round(decimals=3))
print('\ncausal_mask=False (head 0) — 미래 위치도 봅니다')
print(w_off[0, 0].round(decimals=3))

In [ ]:
# 미래 token을 바꿔도 과거 출력이 불변인가
x1 = torch.randn(1, 8, 8)
x2 = x1.clone(); x2[0, 5:] = torch.randn(3, 8)
print('과거(0~4) 출력이 동일한가:', torch.allclose(mha(x1)[0, :5], mha(x2)[0, :5], atol=1e-5))

In [ ]:
# transpose 후 view가 실패하는 이유: memory layout이 연속적이지 않습니다
t = torch.randn(2, 3, 4, 5).transpose(1, 2)            # (2, 4, 3, 5)
print('is_contiguous:', t.is_contiguous())
try:
    t.view(2, 4, 15)
except RuntimeError as e:
    print('view 오류:', str(e)[:80], '...')
print('contiguous() 후 view:', tuple(t.contiguous().view(2, 4, 15).shape))

In [ ]:
selfcheck(5)

---
## Day 6 — LayerNorm, FeedForward, TransformerBlock

**먼저 읽을 교재**: 4.1~4.5

핵심 질문: attention 외에 왜 정규화, 비선형 변환, shortcut(residual) 연결이 필요한가?

**구현**: `LayerNorm.forward`, `FeedForward.__init__`의 `self.layers`, `TransformerBlock.forward`

```text
x -> norm1 -> att -> drop_shortcut -> (+ x)    shortcut 연결
  -> norm2 -> ff  -> drop_shortcut -> (+ 앞 결과)
```

GELU(교재 4.3의 tanh 근사식)와 LayerNorm·TransformerBlock 생성자는 제공됩니다.

In [ ]:
run_pytest('tests/test_model.py', k='LayerNorm or FeedForward or TransformerBlock')

In [ ]:
# LayerNorm은 batch가 아니라 마지막 feature 차원을 정규화합니다
from model import LayerNorm, FeedForward, TransformerBlock

ln = LayerNorm(8); ln.eval()
z = torch.randn(2, 3, 8) * 5 + 10
print('입력 평균      :', z.mean(-1).flatten().round(decimals=2).tolist())
print('출력 평균      :', ln(z).mean(-1).flatten().round(decimals=4).tolist())
print('출력 표준편차  :', ln(z).std(-1, unbiased=False).flatten().round(decimals=3).tolist())
print('nn.LayerNorm과 일치:', torch.allclose(ln(z), torch.nn.LayerNorm(8)(z), atol=1e-5))

In [ ]:
# attention은 token 사이 정보를 섞고, FeedForward는 각 token을 독립적으로 변환합니다
cfg8 = dict(SMOKE_CONFIG, emb_dim=8, n_heads=2, drop_rate=0.0)
ff = FeedForward(cfg8)
a = torch.randn(1, 4, 8)
b = a.clone(); b[0, 0] = torch.randn(8)                  # 위치 0만 변경
print('FFN  : 위치 1~3 출력 동일 =', torch.allclose(ff(a)[0, 1:], ff(b)[0, 1:]))
print('MHA  : 위치 1~3 출력 동일 =', torch.allclose(mha(a)[0, 1:], mha(b)[0, 1:]))

# block은 표현 크기를 유지하고, attention과 FFN 양쪽으로 gradient가 흐릅니다
blk = TransformerBlock(cfg8)
zz = torch.randn(1, 4, 8, requires_grad=True)
out = blk(zz)
print('\nshape 동일:', out.shape == zz.shape, '| 값 동일:', torch.allclose(out, zz))
out.sum().backward()
for name in ['att.W_query.weight', 'ff.layers.0.weight']:
    g = dict(blk.named_parameters())[name].grad
    print(f'{name:20s} grad norm = {g.norm():.4f}')

In [ ]:
selfcheck(6)

---
## Day 7 — GPTModel, language modeling loss, greedy generation

**먼저 읽을 교재**: 4.6~4.7, 5.1(5.1.1~5.1.3), 5.2, 5.4

핵심 질문: token ID는 어떻게 vocabulary logits가 되며 next-token loss는 어떻게 계산하는가?

**구현**: `GPTModel.forward`, `generate_text_simple`, `src/train.py`의 `calc_loss_batch`

```text
token IDs -> tok_emb + pos_emb -> drop_emb -> trf_blocks -> final_norm
          -> out_head -> logits (B, T, V) -> calc_loss_batch -> loss
```

검증 순서: 초기 loss -> tiny-batch overfit -> Smoke 학습 -> greedy 생성

In [ ]:
run_pytest('tests/test_model.py', 'tests/test_train.py')

In [ ]:
# text에서 logits까지의 tensor 흐름
from model import GPTModel, generate_text_simple

m = GPTModel(SMOKE_CONFIG).to(DEVICE); m.eval()
shape_trace(m, batch_size=2, seq_len=8, device=DEVICE)
print()
model_summary(m, SMOKE_CONFIG)

In [ ]:
# 검증 1 — 학습 전 loss가 ln(vocab_size) 근처인가
from train import calc_loss_batch

V = SMOKE_CONFIG['vocab_size']
idx = torch.randint(0, V, (4, 16), device=DEVICE)
tgt = torch.randint(0, V, (4, 16), device=DEVICE)
with torch.no_grad():
    logits = m(idx)
    loss = calc_loss_batch(idx, tgt, m, DEVICE)
print(f'초기 loss {loss.item():.3f}  |  ln({V}) = {math.log(V):.3f}')
print('logits shape:', tuple(logits.shape), '<- softmax 전의 점수입니다')

In [ ]:
# 검증 2 — tiny-batch overfit: 작은 batch 하나를 반복하면 loss가 뚜렷하게 줄어야 합니다
set_seed(0)
mm = GPTModel(SMOKE_CONFIG).to(DEVICE); mm.train()
opt = torch.optim.AdamW(mm.parameters(), lr=3e-3)
xb = torch.randint(0, V, (2, 8), device=DEVICE)
yb = torch.randint(0, V, (2, 8), device=DEVICE)
for step in range(200):
    opt.zero_grad()
    l = calc_loss_batch(xb, yb, mm, DEVICE)
    l.backward()
    opt.step()
    if step == 0:
        first = l.item()
    if step % 50 == 0:
        print(f'step {step:>3}  loss {l.item():.4f}')
print(f'\n{first:.3f} -> {l.item():.4f}  (초기의 30% 이하이면 계산 graph가 연결된 것입니다)')

In [ ]:
# 검증 3 — Smoke end-to-end: 전체 pipeline이 끝까지 실행되는가
from bpe import BPETokenizer
from train import (train_model_simple, plot_losses, save_checkpoint, generate,
                   text_to_token_ids, token_ids_to_text)

smoke_tok = BPETokenizer(vocab_size=SMOKE_DATA['vocab_size']).train(corpus[:SMOKE_DATA['chars']])
# smoke_tok = BPETokenizer(vocab_size=300).load(SMOKE_DATA['vocab_path'])   # 제공 vocabulary 사용 시
assert_vocab_match(SMOKE_CONFIG, smoke_tok)

ctx = SMOKE_CONFIG['context_length']
tr = create_dataloader_v1(corpus[:SMOKE_DATA['chars']], smoke_tok, batch_size=SMOKE_TRAIN['batch_size'],
                          max_length=ctx, stride=ctx, drop_last=True, shuffle=True)
va = create_dataloader_v1(val_corpus[:SMOKE_DATA['chars'] // 5], smoke_tok,
                          batch_size=SMOKE_TRAIN['batch_size'], max_length=ctx, stride=ctx,
                          drop_last=False, shuffle=False)
print(f'train {len(tr.dataset)}개 sample, {len(tr)} steps/epoch | val {len(va.dataset)}개 sample')

set_seed(42)
smoke_model = GPTModel(SMOKE_CONFIG).to(DEVICE)
opt = torch.optim.AdamW(smoke_model.parameters(), lr=SMOKE_TRAIN['learning_rate'],
                        weight_decay=SMOKE_TRAIN['weight_decay'])
smoke_hist = train_model_simple(smoke_model, tr, va, opt, DEVICE,
                                num_epochs=SMOKE_TRAIN['num_epochs'],
                                eval_freq=SMOKE_TRAIN['eval_freq'], eval_iter=SMOKE_TRAIN['eval_iter'],
                                start_context='이 영화는', tokenizer=smoke_tok,
                                grad_clip=SMOKE_TRAIN['grad_clip'])
save_checkpoint(smoke_model, opt, 'data/smoke_ckpt.pth')

train_losses, val_losses, tokens_seen = smoke_hist
epochs_seen = torch.linspace(0, SMOKE_TRAIN['num_epochs'], len(train_losses))
plot_losses(epochs_seen, tokens_seen, train_losses, val_losses, path='data/loss_smoke.png');

In [ ]:
# 검증 4 — greedy 생성이 context window를 넘지 않고 token을 이어 붙이는가
prompt = text_to_token_ids('이 영화는', smoke_tok).to(DEVICE)
smoke_model.eval()
out = generate_text_simple(smoke_model, prompt, max_new_tokens=40, context_size=ctx)
print('길이:', prompt.shape[1], '->', out.shape[1])
print(token_ids_to_text(out, smoke_tok))

`src/train.py`의 `train_model_simple`에서 `optimizer.zero_grad() -> loss.backward() -> optimizer.step()` 순서를 찾고,
`evaluate_model`이 평가할 때 `model.eval()`과 `torch.no_grad()`를 사용한 뒤 `model.train()`으로 돌아가는 이유를 설명하세요.

In [ ]:
selfcheck(7, trace=True)

---
## Day 8 — 감성 분류 head와 작은 통합 실행

**먼저 읽을 교재**: 6.1~6.3, 6.5~6.7 (교재의 `[:, -1, :]` 대신 마지막 non-pad 위치 사용)

핵심 질문: 다음 token을 예측하도록 사전 학습한 모델을 문장 분류에 어떻게 재사용하는가?

**구현**: `src/finetune.py`의 `prepare_for_classification`, `last_token_logits`

```text
사전 학습한 GPTModel -> 전체 동결 -> out_head를 Linear(emb_dim, 2)로 교체
                     -> 마지막 TransformerBlock과 final_norm만 다시 학습 가능
review IDs -> model -> logits (B, T, 2) -> 마지막 유효 token 위치 -> (B, 2)
```

순서: 분류 테스트 -> Smoke backbone으로 train·eval loop 확인 -> Light 사전 학습과 분류 -> 여건에 따라 Basic

In [ ]:
run_pytest('tests/test_finetune.py')

In [ ]:
from torch.utils.data import DataLoader
from finetune import (load_sentiment_jsonl, NSMCDataset, prepare_for_classification,
                      last_token_logits, train_classifier_simple, classify_review)
from train import load_checkpoint

def sentiment_loaders(tok, max_length, n_train, n_val, batch_size=16):
    pad = tok.get_pad_id()
    train_rows = load_sentiment_jsonl('data/nsmc_sentiment_train.jsonl', limit=n_train)
    val_rows = load_sentiment_jsonl('data/nsmc_sentiment_val.jsonl', limit=n_val)
    tr_dl = DataLoader(NSMCDataset(train_rows, tok, max_length=max_length, pad_token_id=pad),
                       batch_size=batch_size, shuffle=True, drop_last=True)
    va_dl = DataLoader(NSMCDataset(val_rows, tok, max_length=max_length, pad_token_id=pad),
                       batch_size=64, shuffle=False)
    return tr_dl, va_dl, val_rows

def run_sentiment(backbone, tok, cfg, n_train, n_val, epochs, lr=5e-4):
    # max_length는 backbone의 context_length를 넘으면 안 됩니다
    tr_dl, va_dl, val_rows = sentiment_loaders(tok, cfg['context_length'], n_train, n_val)
    model = prepare_for_classification(backbone, num_classes=2).to(DEVICE)
    total = sum(p.numel() for p in model.parameters())
    print(f'학습 parameter {count_parameters(model):,} / 전체 {total:,}')
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    train_classifier_simple(model, tr_dl, va_dl, opt, DEVICE, num_epochs=epochs,
                            eval_freq=50, eval_iter=5, pad_token_id=tok.get_pad_id())
    return model, val_rows

def show_predictions(model, tok, rows, max_length, n=5):
    for r in rows[:n]:
        pred = classify_review(r['text'], model, tok, DEVICE, max_length=max_length,
                               pad_token_id=tok.get_pad_id())
        print(f"정답 {'긍정' if r['label'] else '부정'} | 예측 {pred} | {r['text'][:40]}")

In [ ]:
# Smoke backbone + 작은 subset으로 train·eval loop 확인
smoke_backbone = GPTModel(SMOKE_CONFIG).to(DEVICE)
load_checkpoint(smoke_backbone, None, 'data/smoke_ckpt.pth', DEVICE)

set_seed(42)
smoke_clf, val_rows = run_sentiment(smoke_backbone, smoke_tok, SMOKE_CONFIG,
                                    n_train=2000, n_val=500, epochs=2)
show_predictions(smoke_clf, smoke_tok, val_rows, SMOKE_CONFIG['context_length'])

# padding 길이가 달라도 같은 문장의 결과는 같아야 합니다
ids = smoke_tok.encode('정말 재미있는 영화')[:20]
pad = smoke_tok.get_pad_id()
a = torch.tensor([ids + [pad] * 2], device=DEVICE)
b = torch.tensor([ids + [pad] * 8], device=DEVICE)
smoke_clf.eval()
with torch.no_grad():
    print('\n마지막 non-pad 위치 :', torch.allclose(last_token_logits(smoke_clf, a, pad),
                                                last_token_logits(smoke_clf, b, pad), atol=1e-5))
    print('교재의 [:, -1, :]   :', torch.allclose(smoke_clf(a)[:, -1, :], smoke_clf(b)[:, -1, :], atol=1e-5))

### Light 사전 학습

Smoke가 안정적으로 실행되면 Light를 이어서 실행합니다. 제공된 Light vocabulary를 사용합니다.
CPU에서도 실행되지만 GPU 런타임이면 더 빠릅니다.

In [ ]:
light_tok = BPETokenizer(vocab_size=LIGHT_DATA['vocab_size']).load(LIGHT_DATA['vocab_path'])
# 직접 학습해 비교하려면: light_tok = BPETokenizer(LIGHT_DATA['vocab_size']).train(corpus[:LIGHT_DATA['chars']])
assert_vocab_match(LIGHT_CONFIG, light_tok)

ctx = LIGHT_CONFIG['context_length']
t0 = time.time()
tr = create_dataloader_v1(corpus[:LIGHT_DATA['chars']], light_tok, batch_size=LIGHT_TRAIN['batch_size'],
                          max_length=ctx, stride=ctx, drop_last=True, shuffle=True)
va = create_dataloader_v1(val_corpus[:LIGHT_DATA['chars'] // 10], light_tok,
                          batch_size=LIGHT_TRAIN['batch_size'], max_length=ctx, stride=ctx,
                          drop_last=False, shuffle=False)
print(f'encode {time.time()-t0:.1f}초 | train {len(tr.dataset):,}개 / val {len(va.dataset):,}개 sample')

set_seed(42)
light_model = GPTModel(LIGHT_CONFIG).to(DEVICE)
print(f'파라미터 {count_parameters(light_model):,} | {len(tr)} steps/epoch')
opt = torch.optim.AdamW(light_model.parameters(), lr=LIGHT_TRAIN['learning_rate'],
                        weight_decay=LIGHT_TRAIN['weight_decay'])

t0 = time.time()
light_hist = train_model_simple(light_model, tr, va, opt, DEVICE,
                                num_epochs=LIGHT_TRAIN['num_epochs'],
                                eval_freq=LIGHT_TRAIN['eval_freq'], eval_iter=LIGHT_TRAIN['eval_iter'],
                                start_context='이 영화는', tokenizer=light_tok,
                                grad_clip=LIGHT_TRAIN['grad_clip'], ckpt_path='data/light_ckpt.pth')
print(f'총 {time.time()-t0:.0f}초')
tl, vl, seen = light_hist
plot_losses(torch.linspace(0, LIGHT_TRAIN['num_epochs'], len(tl)), seen, tl, vl,
            path='data/loss_light.png');

In [ ]:
# Light backbone으로 감성 분류
light_backbone = GPTModel(LIGHT_CONFIG).to(DEVICE)
load_checkpoint(light_backbone, None, 'data/light_ckpt.pth', DEVICE)

set_seed(42)
light_clf, val_rows = run_sentiment(light_backbone, light_tok, LIGHT_CONFIG,
                                    n_train=4000, n_val=800, epochs=3)
show_predictions(light_clf, light_tok, val_rows, LIGHT_CONFIG['context_length'])
# 정확도 임계값은 없습니다. 실행 조건, loss, 예측 예시, 관찰 정확도를 REPORT.md에 기록하세요.

### Basic (여건이 허락하는 범위)

Basic은 Light 이후 진행 상황, GPU 사용 가능 여부, 런타임 제한을 고려해 실행합니다.
Python으로 pair 빈도를 매번 다시 세는 단순 BPE는 이 규모에서 오래 걸립니다.
Apple Silicon 노트북 CPU 기준으로 vocabulary 학습에 약 9분, encode에 약 2~5분이 걸렸으며 Colab CPU에서는 더 오래 걸릴 수 있습니다.
한 번 학습한 vocabulary는 `BASIC_DATA['vocab_path']`에 저장되어 다음부터 재사용됩니다.
실행하려면 `RUN_BASIC = True`로 바꾸세요.

In [ ]:
RUN_BASIC = False

if RUN_BASIC:
    basic_tok = BPETokenizer(vocab_size=BASIC_DATA['vocab_size'])
    if Path(BASIC_DATA['vocab_path']).exists():
        basic_tok.load(BASIC_DATA['vocab_path'])
    else:
        t0 = time.time()
        basic_tok.train(corpus[:BASIC_DATA['chars']]).save(BASIC_DATA['vocab_path'])
        print(f'BPE 학습 {time.time()-t0:.0f}초')
    assert_vocab_match(BASIC_CONFIG, basic_tok)

    ctx = BASIC_CONFIG['context_length']
    t0 = time.time()
    tr = create_dataloader_v1(corpus[:BASIC_DATA['chars']], basic_tok, batch_size=BASIC_TRAIN['batch_size'],
                              max_length=ctx, stride=ctx, drop_last=True, shuffle=True)
    va = create_dataloader_v1(val_corpus, basic_tok, batch_size=BASIC_TRAIN['batch_size'],
                              max_length=ctx, stride=ctx, drop_last=False, shuffle=False)
    print(f'encode {time.time()-t0:.0f}초 | train {len(tr.dataset):,}개 / val {len(va.dataset):,}개 sample')

    set_seed(42)
    basic_model = GPTModel(BASIC_CONFIG).to(DEVICE)
    print(f'파라미터 {count_parameters(basic_model):,} | {len(tr)} steps/epoch')
    opt = torch.optim.AdamW(basic_model.parameters(), lr=BASIC_TRAIN['learning_rate'],
                            weight_decay=BASIC_TRAIN['weight_decay'])
    basic_hist = train_model_simple(basic_model, tr, va, opt, DEVICE,
                                    num_epochs=BASIC_TRAIN['num_epochs'],
                                    eval_freq=BASIC_TRAIN['eval_freq'], eval_iter=BASIC_TRAIN['eval_iter'],
                                    start_context='이 영화는', tokenizer=basic_tok,
                                    grad_clip=BASIC_TRAIN['grad_clip'], ckpt_path='data/basic_ckpt.pth')
    tl, vl, seen = basic_hist
    plot_losses(torch.linspace(0, BASIC_TRAIN['num_epochs'], len(tl)), seen, tl, vl,
                path='data/loss_basic.png')

In [ ]:
selfcheck(8)

---
## 마무리 — 전체 테스트와 REPORT 기록

모든 Day를 마친 뒤 전체 테스트와 개념 질문을 확인하고, 아래 요약을 `REPORT.md` 4장에 옮깁니다.
완료한 규모의 생성 예시 하나와 감성 분류의 예측 결과도 함께 기록합니다.

In [ ]:
selfcheck()

In [ ]:
# REPORT.md 4장 기록용 요약
def summarize(name, cfg, model, hist):
    tl, vl, seen = hist
    first = f'{tl[0]:.3f}/{vl[0]:.3f}' if tl else '-'
    last = f'{tl[-1]:.3f}/{vl[-1]:.3f}' if tl else '-'
    print(f"{name:6s} vocab {cfg['vocab_size']:>5} ctx {cfg['context_length']:>4} "
          f"params {count_parameters(model):>10,} tokens seen {seen[-1] if seen else 0:>9,} "
          f"train/val {first} -> {last}")

summarize('Smoke', SMOKE_CONFIG, smoke_model, smoke_hist)
summarize('Light', LIGHT_CONFIG, light_model, light_hist)
if RUN_BASIC:
    summarize('Basic', BASIC_CONFIG, basic_model, basic_hist)

prompt = text_to_token_ids('이 영화는', light_tok).to(DEVICE)
out = generate(light_model, prompt, 40, LIGHT_CONFIG['context_length'], temperature=0.0)
print('\n[Light greedy 생성]', token_ids_to_text(out, light_tok))

### 선택 심화 — temperature와 top-k

필수 항목을 마친 뒤 진행합니다. 같은 checkpoint와 같은 prompt로 생성 방식만 바꿔 관찰합니다.

In [ ]:
logits = torch.tensor([3.0, 2.0, 1.0, 0.5, 0.1, -1.0])
for T in [0.2, 1.0, 2.0]:
    print(f'T={T:<4}', torch.softmax(logits / T, dim=-1).round(decimals=3).tolist())

prompt = text_to_token_ids('이 영화는', light_tok).to(DEVICE)
for name, kw in [('greedy', dict(temperature=0.0)),
                 ('T=0.8', dict(temperature=0.8)),
                 ('T=0.8 top-k=20', dict(temperature=0.8, top_k=20))]:
    set_seed(0)
    out = generate(light_model, prompt, 40, LIGHT_CONFIG['context_length'], **kw)
    print(f'\n[{name}] {token_ids_to_text(out, light_tok)}')